# WavLM Embedding Extraction - Colab
GPU: T4 (free) | Runtime: ~45 min for 14,998 utterances

In [ ]:
# STEP 1: Setup
!pip install transformers datasets librosa accelerate torch kaggle -q
!mkdir -p /root/.kaggle
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write('{"username":"subhajitdas","key":"YOUR_KAGGLE_KEY"}')
import os, json, time, numpy as np, torch
import librosa
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
from tqdm.auto import tqdm
print('Setup done', flush=True)


In [ ]:
# STEP 2: Download Kaggle dataset
!kaggle datasets download -d subhajitdas/chuckle-net-prosody-fusion -p /content/chuckle-net --force
!cd /content/chuckle-net && unzip -o chuckle-net-prosody-fusion.zip
DATASET_DIR = '/content/chuckle-net'
AUDIO_DIR = os.path.join(DATASET_DIR, 'audio')
PROSODY_PATH = os.path.join(DATASET_DIR, 'prosody_phaseD.json')
ALIGNED_PATH = os.path.join(DATASET_DIR, 'aligned_utterances.jsonl')
print(f'Audio files: {len(os.listdir(AUDIO_DIR))}', flush=True)


In [ ]:
# STEP 3: Load WavLM on GPU
device = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}', flush=True)
wavlm = WavLMModel.from_pretrained('microsoft/wavlm-base-plus').to(device)
wavlm.eval()
fext = Wav2Vec2FeatureExtractor.from_pretrained('microsoft/wavlm-base-plus')
SR = 16000
DUR = 5.0
MAX_LEN = int(DUR * SR)
print('WavLM loaded on GPU', flush=True)


In [ ]:
# STEP 4: Load aligned utterances
aligned_data = [json.loads(l) for l in open(ALIGNED_PATH)]
print(f'Total utterances: {len(aligned_data)}', flush=True)

utt_metadata = {}
for d in aligned_data:
    uid = f"{d['video_id']}_{d['start']:.2f}"
    utt_metadata[uid] = {
        'video_id': d['video_id'],
        'start': d['start'],
        'end': d['end'],
        'label': d.get('label', 0),
        'word': d.get('word', ''),
        'context_words': d.get('context_words', [])
    }
print(f'Unique utterances: {len(utt_metadata)}', flush=True)


In [ ]:
# STEP 5: Extract embeddings
def extract_embedding(audio_path, start, duration=DUR):
    try:
        off = max(0, start - 0.05)
        audio, _ = librosa.load(audio_path, sr=SR, mono=True, offset=off, duration=duration)
        if len(audio) < MAX_LEN: audio = np.pad(audio, (0, MAX_LEN - len(audio)))
        audio = audio[:MAX_LEN]
        wav_np = [audio]
        inputs = fext(wav_np, sampling_rate=SR, return_tensors='pt', padding=False)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            hidden = wavlm(**inputs).last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
        return hidden
    except:
        return np.zeros(768, dtype=np.float32)

embeddings = {}
audio_files = [f for f in os.listdir(AUDIO_DIR) if f.endswith('.mp3')]
print(f'Processing {len(audio_files)} files, {len(utt_metadata)} utterances...', flush=True)

for audio_file in tqdm(audio_files, desc='Audio files'):
    vid = audio_file.replace('.mp3', '')
    audio_path = os.path.join(AUDIO_DIR, audio_file)
    vid_utts = [(uid, u) for uid, u in utt_metadata.items() if u['video_id'] == vid]
    for uid, utt in vid_utts:
        emb = extract_embedding(audio_path, utt['start'])
        embeddings[uid] = {
            'embedding': emb.tolist(),
            'video_id': vid,
            'start': utt['start'],
            'end': utt['end'],
            'label': utt['label'],
            'word': utt.get('word', ''),
            'context_words': utt.get('context_words', [])
        }
    if len(embeddings) % 1000 == 0:
        print(f'  Extracted {len(embeddings)} embeddings...', flush=True)

print(f'Extracted embeddings for {len(embeddings)} utterances', flush=True)


In [ ]:
# STEP 6: Save embeddings
OUTPUT_PATH = '/content/chuckle-net/wavlm_embeddings.json'
with open(OUTPUT_PATH, 'w') as f:
    json.dump(embeddings, f)
print(f'Saved to {OUTPUT_PATH}', flush=True)
labels = [v['label'] for v in embeddings.values()]
print(f'Positive: {sum(labels)}, Negative: {len(labels)-sum(labels)}', flush=True)
print('DONE!', flush=True)
